In [68]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [92]:
from backend.agents.extractor import Extractor
from backend.llms.ollama import OllamaLLM, OpenAIOutputMessage, LlamaOutputMessage
from backend.llms.bedrock import BedrockNova
import logging
from backend.utils import setup_logger
from backend.prompt_hub import PromptHub

setup_logger(logging.DEBUG)

# openai_llm = OllamaLLM(model_id="gpt-oss:20b", OutputMessage=OpenAIOutputMessage)
llama_llm = OllamaLLM(model_id="llama3.2:1b", OutputMessage=LlamaOutputMessage)
# nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

✅ Development logging enabled (DEBUG level) - Console


# test pitching agent

In [109]:
# from backend.agents.ai_sales_coaching.pitching_data_model import PitchingRecommend
# from backend.agents.ai_sales_coaching.extract_data_model import Guide
from backend.prompt_hub import PromptHub

transition_closing_agent = Extractor(
    agent_name="pitching_agent",
    llm=llama_llm,
    # system_prompt=PromptHub().pitch_agent,
    system_prompt=PromptHub().transition_closing,
    # DataModel=PitchingRecommend,
    # DataModel=Guide,
    format="json",
    max_retries=2,
)

TypeError: Extractor.__init__() missing 1 required positional argument: 'DataModel'

In [110]:
print(PromptHub().transition_closing)

# CUSTOMER INTEREST DETECTION

Analyze 8-second audio chunk from ongoing insurance sales conversation.

Return `true` ONLY if customer shows interest through trigger words:

**Interest Triggers:**
- "สนใจ" (interested)
- "อยากรู้" (want to know)
- "ดีจัง" (sounds good)
- "น่าสนใจ" (interesting)
- "ช่วยได้" (can help)
- "เหมาะกับเรา" (suitable for us)
- "ราคาเท่าไหร่" (how much)
- "เริ่มยังไง" (how to start)
- "สมัครได้ไหม" (can I apply)

Return `false` for all other cases.

**Output:** `true` or `false` only.


In [101]:
# Test conversations with mixed sales-customer dialogue
selling_lines = [
    # Price objection scenario
    "แผนประกันนี้คุ้มครองครอบคลุมมากค่ะ เบี้ยเดือนละ 3,000 บาท แพงไปนะครับ ผมไม่ไหวหรอก",
    
    # Confusion about coverage
    "เราคุ้มครองทั้งอุบัติเหตุและเจ็บป่วยนะคะ ไม่เข้าใจเลยครับ ความคุ้มครองมันยุ่งยากมาก งงจริงๆ",
    
    # Existing insurance concern
    "แผนนี้ให้ผลตอบแทนดีมากค่ะ ผมมีประกันแล้วครับ ทำกับบริษัทอื่นไว้ เคลมได้ไหมยังไม่รู้",
    
    # Trust and claim issues
    "บริษัทเรามีประสบการณ์ 50 ปีค่ะ บริษัทนี้ไม่รู้จักเลยครับ เคลมยากไหม เคยได้ยินว่าโดนปฏิเสธ",
    
    # Age and health concerns
    "เรารับประกันถึงอายุ 70 ปีค่ะ ผมอายุมากแล้ว 65 แล้ว มีโรคเบาหวาน ยังทำได้ไหม",
    
    # Family resistance
    "แผนนี้ช่วยคุ้มครองครอบครัวด้วยค่ะ ครอบครัวไม่เห็นด้วยครับ ภรรยาบอกว่าลูกยังเล็ก ค่าใช้จ่ายเยอะ",
    
    # Young age overconfidence
    "การป้องกันตั้งแต่หนุ่มสาวสำคัญมากค่ะ ผมยังหนุ่ม ไม่เจ็บไม่ป่วย ไม่เร่งด่วนหรอกครับ",
    
    # Avoidance behavior
    "มีคำถามอะไรเพิ่มเติมไหมคะ เดี๋ยวผมโทรกลับนะครับ ตอนนี้ไม่ว่าง รีบไปประชุม",
    
    # Financial constraints
    "เรามีแผนผ่อนสะดวกค่ะ เงินเดือนน้อย มีหนี้เยอะ ค่าใช้จ่ายประจำเดือนเต็มแล้ว",
    
    # Pre-existing condition worry
    "เราคุ้มครองโรคใหม่ที่เกิดขึ้นค่ะ โรคเก่าไม่คุ้มครองใช่ไหม งั้นไม่คุ้มสำหรับผม",
    
    # Claim difficulty fear
    "ขั้นตอนเคลมง่ายมากค่ะ แค่โทรแจ้ง เคลมยากไหมครับ เคยได้ยินว่าต้องรอนาน",
    
    # No negative signals - positive response
    "นี่คือรายละเอียดแผนค่ะ ฟังดูน่าสนใจครับ อยากทราบรายละเอียดเพิ่มเติม"
]

In [106]:
test_dict = {}
for i in range(len(selling_lines)):
    text = f"TEXT: {selling_lines[i]}"
    print(text)
    result = pitching_agent.run([{"role": "user", "content": text}])
    print(result)
    test_dict[text] = result

2025-11-26 22:47:32,933 - pitching_agent - run:64 - INFO - start extracting...


TEXT: แผนประกันนี้คุ้มครองครอบคลุมมากค่ะ เบี้ยเดือนละ 3,000 บาท แพงไปนะครับ ผมไม่ไหวหรอก


2025-11-26 22:47:36,849 - pitching_agent - run:74 - WARNING - attempt 1 failed: Expecting value: line 1 column 1 (char 0)
2025-11-26 22:47:36,850 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-26 22:47:41,627 - pitching_agent - run:74 - WARNING - attempt 2 failed: Expecting value: line 1 column 1 (char 0)
2025-11-26 22:47:41,627 - pitching_agent - run:87 - ERROR - all 2 attempts failed
2025-11-26 22:47:41,628 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-26 22:47:41,628 - pitching_agent - run:64 - INFO - start extracting...


action=None explanation=None signals=[] lines_to_say=[]
TEXT: เราคุ้มครองทั้งอุบัติเหตุและเจ็บป่วยนะคะ ไม่เข้าใจเลยครับ ความคุ้มครองมันยุ่งยากมาก งงจริงๆ


2025-11-26 22:47:46,195 - pitching_agent - run:74 - WARNING - attempt 1 failed: Expecting value: line 1 column 1 (char 0)
2025-11-26 22:47:46,195 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-26 22:47:57,744 - pitching_agent - run:74 - WARNING - attempt 2 failed: Expecting value: line 1 column 1 (char 0)
2025-11-26 22:47:57,744 - pitching_agent - run:87 - ERROR - all 2 attempts failed
2025-11-26 22:47:57,745 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-26 22:47:57,745 - pitching_agent - run:64 - INFO - start extracting...


...

action=None explanation=None signals=[] lines_to_say=[]
TEXT: ขั้นตอนเคลมง่ายมากค่ะ แค่โทรแจ้ง เคลมยากไหมครับ เคยได้ยินว่าต้องรอนาน


2025-11-26 22:48:52,412 - pitching_agent - run:74 - WARNING - attempt 1 failed: Expecting value: line 1 column 1 (char 0)
2025-11-26 22:48:52,413 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-26 22:48:55,150 - pitching_agent - run:74 - WARNING - attempt 2 failed: Expecting value: line 1 column 1 (char 0)
2025-11-26 22:48:55,150 - pitching_agent - run:87 - ERROR - all 2 attempts failed
2025-11-26 22:48:55,151 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-26 22:48:55,151 - pitching_agent - run:64 - INFO - start extracting...


action=None explanation=None signals=[] lines_to_say=[]
TEXT: นี่คือรายละเอียดแผนค่ะ ฟังดูน่าสนใจครับ อยากทราบรายละเอียดเพิ่มเติม


2025-11-26 22:49:00,067 - pitching_agent - run:74 - WARNING - attempt 1 failed: Expecting value: line 1 column 1 (char 0)
2025-11-26 22:49:00,067 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-26 22:49:02,926 - pitching_agent - run:74 - WARNING - attempt 2 failed: Expecting value: line 1 column 1 (char 0)
2025-11-26 22:49:02,926 - pitching_agent - run:87 - ERROR - all 2 attempts failed
2025-11-26 22:49:02,926 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


action=None explanation=None signals=[] lines_to_say=[]


In [107]:
test_dict

{'TEXT: แผนประกันนี้คุ้มครองครอบคลุมมากค่ะ เบี้ยเดือนละ 3,000 บาท แพงไปนะครับ ผมไม่ไหวหรอก': Guide(action=None, explanation=None, signals=[], lines_to_say=[]),
 'TEXT: เราคุ้มครองทั้งอุบัติเหตุและเจ็บป่วยนะคะ ไม่เข้าใจเลยครับ ความคุ้มครองมันยุ่งยากมาก งงจริงๆ': Guide(action=None, explanation=None, signals=[], lines_to_say=[]),
 'TEXT: แผนนี้ให้ผลตอบแทนดีมากค่ะ ผมมีประกันแล้วครับ ทำกับบริษัทอื่นไว้ เคลมได้ไหมยังไม่รู้': Guide(action='สั่งให้มอบหมายงานดูแลประกันใหม่', explanation='อธิบายว่าทำไมเป็นความคิดที่ควรจะสั่งให้มอบหมายงานดูแลประกันใหม่คะ เพราะเป็นการรักษาความปลอดภัยดีกว่าไม่ได้', signals=['การแสดงความสนใจในผลตอบแทนที่คุณได้รับ'], lines_to_say=['สิ่งสำคัญคือ มาคลายใจกันก่อนนะคะ เพราะเป็นการรักษาความปลอดภัยดีกว่า']),
 'TEXT: บริษัทเรามีประสบการณ์ 50 ปีค่ะ บริษัทนี้ไม่รู้จักเลยครับ เคลมยากไหม เคยได้ยินว่าโดนปฏิเสธ': Guide(action='ไม่เชื่อจากบริษัท', explanation='บริษัทนี้ไม่มีความเกี่ยวข้องในวงการประกันค่ะ ไม่รู้จักเลย', signals=['ไม่รู้จักจากบริษัท ซึ่งทำให้ไม่สามารถให้บริการที่ดีค

# transition close

In [13]:
# Create a sales stage classifier
from backend.agents.ai_sales_coaching.extract_data_model import ClassifiedStage
stage_classifier = Extractor(
    agent_name="stage_classifier",
    llm=llama_llm,
    system_prompt=PromptHub().classify_sales_stage,
    DataModel=ClassifiedStage,
    format="json",
)

# Test with different conversation examples
# test_conversations = [
#     "สวัสดีครับผมอนันต์จากบริษัทประกันภัย เรียนสายคุยสมชาย ไม่ทราบว่าสะดวกคุยไหมครับ?",
#     "ไม่ทราบว่าคุณสมชายมีเป้าหมายทางการเงินอะไรบ้างครับ?",
#     "เรามีแผนประกันที่เหมาะกับคุณมาก ให้ผลตอบแทน 6% ต่อปี",
#     "คุณพร้อมที่จะเริ่มต้นแผนนี้ไหมครับ?"
# ]

# for conversation in test_conversations:
#     result = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
#     print(f"Conversation: {conversation[:50]}...")
#     print(f"Stage: {result.stage}")
#     print(f"Signals: {result.signals}")
#     print("---")
# response = nova_llm.run(
#     system_prompt=PromptHub().classify_sales_stage,
#     messages=[dict(role="user", content=test_conversations[0])]
# )

In [14]:
conversation = """\
สวัสดีครับผมอนันต์จากบริษัทประกันชีวิต เรียนสายคุณสมชาย พอจะสะดวกคุยซักห้านาทีไหมครับ? สะดวกครับ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-25 20:37:56,489 - stage_classifier - run:64 - INFO - start extracting...
2025-11-25 20:38:01,051 - stage_classifier - run:74 - WARNING - attempt 1 failed: Expecting value: line 1 column 1 (char 0)
2025-11-25 20:38:01,051 - stage_classifier - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-25 20:38:04,521 - stage_classifier - run:74 - WARNING - attempt 2 failed: Expecting value: line 1 column 1 (char 0)
2025-11-25 20:38:04,522 - stage_classifier - run:87 - ERROR - all 2 attempts failed
2025-11-25 20:38:04,522 - stage_classifier - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


stage=<StageType.UNKNOWN: 'Unknown'> signals=[]
{'input_tokens': 0, 'output_tokens': 0, 'input_cost': 0, 'output_cost': 0, 'total_cost': 0}


In [15]:
conversation = """\
สะดวกคุยซักห้านาทีไหมครับ? สะดวกครับมีเรื่องอะไรหรือครับ? ผมขอสอบถามพี่สมชายว่าไม่แน่ใจว่าตอนนี้
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-25 20:38:09,531 - stage_classifier - run:64 - INFO - start extracting...
2025-11-25 20:38:15,248 - stage_classifier - run:74 - WARNING - attempt 1 failed: 1 validation error for ClassifiedStage
stage
  Input should be 'Greeting', 'Discovery', 'Pitch', 'Closing', 'Follow Up', 'Unknown' or 'Objection' [type=enum, input_value='Introduction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/enum
2025-11-25 20:38:15,249 - stage_classifier - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-25 20:38:21,949 - stage_classifier - run:71 - INFO - extraction successful on attempt 2
2025-11-25 20:38:21,949 - stage_classifier - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


stage=<StageType.GREETING: 'Greeting'> signals=['agent introduction', 'company mention']
{'input_tokens': 0, 'output_tokens': 0, 'input_cost': 0, 'output_cost': 0, 'total_cost': 0}


In [9]:
conversation = """\
ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:22,342 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:23,637 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:23,638 - stage_classifier - run:92 - INFO - end extracting with input_tokens=713 and output_tokens=46


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about specific interest in health insurance or critical illness insurance', 'customer response indicating interest in health insurance']
{'input_tokens': 713, 'output_tokens': 46, 'input_cost': 0.0008235149999999999, 'output_cost': 0.00021252, 'total_cost': 0.0010360349999999998}


In [10]:
conversation = """\
อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ ถ้าพี่สมชายสนใจประกันสุขภาพผมแนะนำแผนประกันตัวนี้ครับคุ้มครองในตัวของสุขภาพ ในวงเงินอยู่ที่ห้าพันบาทต่อการรักษาและถ้าในกรณีที่ต้องนอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:23,653 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:26,260 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:26,261 - stage_classifier - run:92 - INFO - end extracting with input_tokens=789 and output_tokens=64


stage=<StageType.PITCH: 'Pitch'> signals=['mentioning specific insurance plan', 'highlighting benefits and coverage', 'showing quantified value (5,000 baht for treatment and 5,000 baht for hospital room)']
{'input_tokens': 789, 'output_tokens': 64, 'input_cost': 0.000911295, 'output_cost': 0.00029568, 'total_cost': 0.0012069749999999999}


In [11]:
conversation = """\
นอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท ไม่แน่ใจว่าใช่ที่พี่สมชายมองหาไหมครับ? ค่ารักษาในแต่ละครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:26,276 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:28,060 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:28,061 - stage_classifier - run:92 - INFO - end extracting with input_tokens=739 and output_tokens=56


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about specific details of hospital room refund', 'inquiring about coverage of treatment costs', 'collecting qualifying data on what conditions are covered']
{'input_tokens': 739, 'output_tokens': 56, 'input_cost': 0.0008535449999999999, 'output_cost': 0.00025872, 'total_cost': 0.0011122649999999999}


In [12]:
conversation = """\
ครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ? ในกรณีค่ารักษาอันนี้ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:28,076 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:29,504 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:29,504 - stage_classifier - run:92 - INFO - end extracting with input_tokens=731 and output_tokens=52


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about coverage details', 'inquiring about specific conditions covered', 'requesting information on what is included in the plan']
{'input_tokens': 731, 'output_tokens': 52, 'input_cost': 0.000844305, 'output_cost': 0.00024024, 'total_cost': 0.001084545}


In [13]:
conversation = """\
เลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น น่าสนใจอยู่นะ ไม่แน่ใจว่าอันนี้ค่าเบี้ยประกันเป็นยังไงบ้าง?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())


2025-11-23 23:51:29,515 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:30,909 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:30,912 - stage_classifier - run:92 - INFO - end extracting with input_tokens=707 and output_tokens=38


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about insurance premium', 'showing interest in the product']
{'input_tokens': 707, 'output_tokens': 38, 'input_cost': 0.000816585, 'output_cost': 0.00017556, 'total_cost': 0.000992145}


In [14]:
conversation = """\
้ค่าเบี้ยประกันเป็นยังไงบ้าง? ค่าเบี้ยประกันตอนตัวนี้อยู่ที่เดือนละห้าร้อยบาทครับ ไม่แน่ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:30,927 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:32,322 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:32,323 - stage_classifier - run:92 - INFO - end extracting with input_tokens=717 and output_tokens=43


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about insurance premium', 'providing specific product details', 'indicating potential interest']
{'input_tokens': 717, 'output_tokens': 43, 'input_cost': 0.0008281349999999999, 'output_cost': 0.00019866, 'total_cost': 0.0010267949999999998}


In [15]:
conversation = """\
ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจถ้างั้นผมต้องทำยังไงบ้าง? ถ้าพี่สมชายสนใจงั้นผมคุยรายละเอียดเรื่องการสมัครเลยไหมครับ?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:32,338 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:34,768 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:34,769 - stage_classifier - run:92 - INFO - end extracting with input_tokens=720 and output_tokens=53


stage=<StageType.CLOSING: 'Closing'> signals=['asking if the customer is interested', 'requesting details on how to proceed if interested', 'inviting to discuss enrollment details']
{'input_tokens': 720, 'output_tokens': 53, 'input_cost': 0.0008315999999999999, 'output_cost': 0.00024486, 'total_cost': 0.00107646}


In [27]:
from backend.agents.ai_sales_coaching.coaching_data_model import SalesCoaching

# Create sales coach
sales_coach = Extractor(
    agent_name="sales_coach",
    llm=llama_llm,
    # llm=openai_llm,
    system_prompt=PromptHub().sales_coaching,
    DataModel=SalesCoaching,
    format="json"
)

# Example usage
# conversation_context = """
# Current Stage: Discovery
# Recent Text: "คุณมีครอบครัวกี่คนครับ? มีสองคนครับ ผมกับภรรยา"
# Customer Data: {"age": 35, "family_size": 2}
# """

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"
""".strip()


conversation_context = """\
Current Stage: Discovery 
Recent Text: "นอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท ไม่แน่ใจว่าใช่ที่พี่สมชายมองหาไหมครับ? ค่ารักษาในแต่ละครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ?"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ? ในกรณีค่ารักษาอันนี้ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น น่าสนใจอยู่นะ ไม่แน่ใจว่าอันนี้ค่าเบี้ยประกันเป็นยังไงบ้าง?"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ค่าเบี้ยประกันเป็นยังไงบ้าง? ค่าเบี้ยประกันตอนตัวนี้อยู่ที่เดือนละห้าร้อยบาทครับ ไม่แน่ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจ"
""".strip()
conversation_context = """\
Current Stage: Discovery 
Recent Text: "ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจถ้างั้นผมต้องทำยังไงบ้าง? ถ้าพี่สมชายสนใจงั้นผมคุยรายละเอียดเรื่องการสมัครเลยไหมครับ?"
""".strip()
response = sales_coach.run([{"role": "user", "content": conversation_context}])
response
# print(f"Next Action: {coaching.next_action}")
# print(f"Suggested Lines: {coaching.suggested_lines}")
# print(f"Stage Transition: {coaching.stage_transition}")


2025-11-25 20:50:52,655 - sales_coach - run:64 - INFO - start extracting...
2025-11-25 20:50:56,734 - sales_coach - run:71 - INFO - extraction successful on attempt 1
2025-11-25 20:50:56,734 - sales_coach - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


SalesCoaching(current_stage='Discovery', next_action='stay/discovery', suggested_lines=['ยังไม่มีแผนการเลยครับ จากที่คุณเล่ามา เรามีแนวทางเหมาะกับคุณมาก'], stage_transition=<StageTransition.STAY: 'stay'>, reason="Need to gather more information about the customer's needs and goals before proceeding")

In [29]:
print(response)

current_stage='Discovery' next_action='stay/discovery' suggested_lines=['ยังไม่มีแผนการเลยครับ จากที่คุณเล่ามา เรามีแนวทางเหมาะกับคุณมาก'] stage_transition=<StageTransition.STAY: 'stay'> reason="Need to gather more information about the customer's needs and goals before proceeding"


In [19]:
conversation_context = """\
Current Stage: Discovery 
Recent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"
""".strip()
print(conversation_context)

Current Stage: Discovery 
Recent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"


In [41]:
sales_coach.calculate_cost()

{'input_tokens': 627,
 'output_tokens': 178,
 'input_cost': 0.000724185,
 'output_cost': 0.00082236,
 'total_cost': 0.0015465449999999999}

In [20]:
response = llama_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

**Context**: Customer has expressed interest in insurance, but there's a slight hesitation.

**COACHING RULES**:

Since the customer has shared clear needs/goals (romantic relationship), it's likely that they are ready to proceed with benefits. However, to ensure a smooth discussion and increase the chances of closing the sale, we need to address their current interest in romantic relationships.

**Next Action**: Ask about their current status regarding romantic relationships and express understanding for any challenges they might be facing.

**Specific Action**: "ครับ/ค่ะ วันนี้พี่สมชายสนใจประกันสุขภาพหรือไม่? มีอะไรอยู่ระหว่างการรักและงานบ้านครั้งใดครับ?"

**Suggested Lines**: "ตอนนี้เรามีอีกอย่างหนึ่งที่ต้องกังวลในด้านการรักและงานบ้านของพี่สมชายแล้วครับ/ค่ะ"


In [22]:
conversation_context

'Current Stage: Discovery \nRecent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"'

In [23]:
response = llama_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

**Current Stage: Discovery**
**Recent Text:** "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"

### Coaching Rule: Discovery Stage

**If missing key data → suggest specific questions**

The customer is clearly interested in their father's health and the agent should ask more targeted questions to gather relevant information.

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_health_issues",
  "suggested_lines": ["คุณพี่สมชายมีปัญหาสุขภาพอยู่หรือยังไม่แน่ใจเกินไปไหมครับ?"],
  "stage_transition": "stay",
  "reason": "Need to understand health issues before recommending insurance solutions"
}
```

### Additional Suggestion

To further assess the customer's interest and needs, ask if they have any specific health concerns or questions about their father's current situation.

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_health_issues",
  "suggested_lines": ["คุณพี่สมชายมีปัญหาสุขภาพอยู

In [19]:
response = openai_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_budget",
  "suggested_lines": [
    "ขอให้บอกหน่อยว่า งบประมาณที่คุณสะดวกต่อการซื้อประกันต่อเดือนประมาณเท่าไหร่ครับ?",
    "ถ้าตั้งใจจะมีการชำระค่างวดประกันอย่างสม่ำเสมอ คุณคาดว่าจะใช้จำนวนเงินเท่าไหร่ต่อเดือน?"
  ],
  "stage_transition": "stay",
  "reason": "ยังขาดข้อมูลสำคัญเกี่ยวกับงบประมาณการซื้อประกัน เพื่อให้สามารถนำเสนอแผนที่เหมาะสมกับลูกค้าได้"
}
```


In [32]:
response = nova_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_insurance_needs",
  "suggested_lines": ["จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?"],
  "stage_transition": "stay",
  "reason": "Need to understand specific insurance needs to tailor the pitch effectively"
}
```


In [33]:
from backend.llms.utils import parse_blockcode
import json
parsed_resposne = parse_blockcode(response.content, "json")
parsed_response = json.loads(parsed_resposne)

In [34]:
print(parsed_resposne)

{
  "current_stage": "Discovery",
  "next_action": "ask_about_insurance_needs",
  "suggested_lines": ["จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?"],
  "stage_transition": "stay",
  "reason": "Need to understand specific insurance needs to tailor the pitch effectively"
}


In [ ]:
SalesCoaching(**parsed_response)[]

SalesCoaching(current_stage='Discovery', next_action='ask_about_insurance_needs', suggested_lines=['จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?'], stage_transition=<StageTransition.STAY: 'stay'>, reason='Need to understand specific insurance needs to tailor the pitch effectively')

In [18]:
from backend.agents.extractor import Extractor
from backend.prompt_hub import PromptHub
from backend.llms.bedrock import BedrockNova
from backend.agents.ai_sales_coaching.extract_data_model import AgentCheckList

nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

agent_checklist_extractor = Extractor(
    agent_name="agent_checklist_extractor",
    llm=nova_llm,
    system_prompt=PromptHub().extract_agent_checklist,
    DataModel=AgentCheckList,
)

In [22]:
# content = """\
# สวัสดีครับผมอนันต์จากบริษัทประกันครับขออนุญาติเรียนสายคุณสมชายเพื่อแนะนำแบบประกันครับ
# """.strip()
# content = """\
# สวัสดีครับผมอนันต์เรียนสายคุณสมชายครับ
# """.strip()
content = """\
เรียนสายคุณสมชาย พอจะสะดวกคุยซักห้านาทีไหมครับ? สะดวกครับ
""".strip()

response = agent_checklist_extractor.run([dict(role="user", content=content)])
response

2025-11-24 21:57:25,379 - agent_checklist_extractor - run:64 - INFO - start extracting...
2025-11-24 21:57:26,632 - agent_checklist_extractor - run:71 - INFO - extraction successful on attempt 1
2025-11-24 21:57:26,633 - agent_checklist_extractor - run:91 - INFO - end extracting with input_tokens=327 and output_tokens=22


AgentCheckList(agent_introduced=None, company_mentioned=None, permission_asked=True)

In [1]:
from backend.llms.bedrock import BedrockNova
from backend.prompt_hub import PromptHub
from backend.agents.extractor import Extractor
from backend.agents.ai_sales_coaching.extract_data_model import Guide

nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

In [9]:
from backend.agents.ai_sales_coaching.extract_data_model import Guide

# 1. GREETING AGENT
greeting_agent = Extractor(
    agent_name="greeting_agent",
    llm=nova_llm,
    system_prompt=PromptHub().greeting_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: สวัสดีครับผมอนันต์จากบริษัทประกันภัยครับ เรียนสายคุณสมชายครับ ไม่ทราบว่าสะดวกคุยไหมครับ? อ๋อ รีบอยู่นะครับ มีแค่ 2-3 นาทีเท่านั้น
""".strip()

response = greeting_agent.run([dict(role="user", content=content)])
print("GREETING AGENT:")
print(response.model_dump_json(indent=4))
greeting_agent.calculate_cost()

GREETING AGENT:
{
    "action": "Ask for brief permission and set time expectation",
    "explanation": "The customer expressed time pressure and gave permission to speak but also indicated a brief time frame. The agent should confirm the time constraint and assure the customer that the conversation will be brief.",
    "signals": [
        "Customer expressed time pressure",
        "Permission granted"
    ],
    "lines_to_say": [
        "ขอบคุณที่รับสาย คุณสมชาย เพราะเวลาของคุณนั้นสำคัญ ฉันจะเริ่มทันทีด้วยเวลาที่เหลือของคุณ 2-3 นาทีนี้ อยากบอกว่าในเรื่องประกันภัยมีสิ่งใหม่ๆ ที่อาจช่วยให้คุณรู้ได้"
    ]
}


{'input_tokens': 579,
 'output_tokens': 229,
 'input_cost': 0.000668745,
 'output_cost': 0.00105798,
 'total_cost': 0.0017267250000000001}

In [10]:
# 2. DISCOVERY AGENT
discovery_agent = Extractor(
    agent_name="discovery_agent",
    llm=nova_llm,
    system_prompt=PromptHub().discovery_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ไม่ทราบว่าคุณสมชายมีครอบครัวไหมครับ? มีครับ มีภรรยาแล้วก็ลูกสองคนครับ ลูกชายกับลูกสาว อายุ 8 กับ 5 ขวบครับ แล้วตอนนี้คุณมีประกันอะไรอยู่บ้างไหมครับ? มีประกันสังคมจากที่ทำงานครับ แต่รู้สึกว่ามันไม่เพียงพอน่ะครับ ถ้าเกิดอะไรขึ้นกับผม ครอบครัวจะอยู่ยังไงครับ
""".strip()

response = discovery_agent.run([dict(role="user", content=content)])
print(response.model_dump_json(indent=4))
discovery_agent.calculate_cost()

{
    "action": "Explore family protection needs",
    "explanation": "The customer has mentioned having a family, which indicates a need for family protection. It's important to delve deeper into their specific concerns and how they perceive the adequacy of their current coverage.",
    "signals": [
        "Customer revealed family situation",
        "Customer expressed concern about current coverage"
    ],
    "lines_to_say": [
        "นั่นแหละ คุณมีภรรยาและลูกสองคน การคุ้มครองครอบครัวของคุณเป็นสิ่งที่สำคัญมาก คุณมีความกังวลอย่างไรเกี่ยวกับประกันสังคมที่คุณมีอยู่ในปัจจุบัน?",
        "คุณมีความกังวลว่าหากเกิดเหตุการณ์ฉุกเฉินอะไรขึ้นกับคุณ ครอบครัวของคุณจะอยู่ได้อย่างไรหรือไม่?"
    ]
}


{'input_tokens': 642,
 'output_tokens': 281,
 'input_cost': 0.00074151,
 'output_cost': 0.0012982199999999999,
 'total_cost': 0.00203973}

In [31]:
# 3. PITCH AGENT  
pitch_agent = Extractor(
    agent_name="pitch_agent",
    llm=llama_llm,
    system_prompt=PromptHub().pitch_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ตามที่คุณสมชายบอกว่าห่วงเรื่องครอบครัว ผมเลยอยากเสนอแบบประกันชีวิตที่จะช่วยดูแลครอบครัวคุณได้ครับ มีความคุ้มครองสูงสุด 5 ล้านบาท เบี้ยประกันเดือนละ 3,500 บาท ไม่เข้าใจครับ ความคุ้มครองหมายถึงอะไร?
""".strip()

response = pitch_agent.run([dict(role="user", content=content)])
print("\nPITCH AGENT:")
print(response.model_dump_json(indent=4))
pitch_agent.calculate_cost()

NameError: name 'Guide' is not defined

In [12]:
# 4. CLOSING AGENT
closing_agent = Extractor(
    agent_name="closing_agent", 
    llm=nova_llm,
    system_prompt=PromptHub().closing_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ถ้าอย่างนั้นคุณสมชายพร้อมที่จะสมัครแบบประกันนี้ไหมครับ? ผมสนใจครับ แต่ขอคุยกับภรรยาก่อนได้ไหม? เพราะเป็นเรื่องใหญ่สำหรับครอบครัว
""".strip()

response = closing_agent.run([dict(role="user", content=content)])
print("\nCLOSING AGENT:")
print(response.model_dump_json(indent=4))
closing_agent.calculate_cost()


CLOSING AGENT:
{
    "action": "Offer to include family in discussion",
    "explanation": "The customer expresses interest but requests to discuss with his spouse first. It's important to respect this and offer a way to include the family in the decision-making process.",
    "signals": [
        "Customer needs family consultation",
        "Customer expresses interest but has a request"
    ],
    "lines_to_say": [
        "แน่นอน คุณสามารถคุยกับภรรยาได้ เราสามารถจัดเวลาพบเจอกับคุณและภรรยาเพื่อให้คุณสามารถแบ่งปันข้อมูลและตัดสินใจร่วมกันได้",
        "คุณสามารถแจ้งให้ภรรยาของคุณทราบว่าเราพร้อมรับสมัครพร้อมกันได้"
    ]
}


{'input_tokens': 538,
 'output_tokens': 241,
 'input_cost': 0.00062139,
 'output_cost': 0.00111342,
 'total_cost': 0.0017348099999999998}

In [13]:
# 5. OBJECTION HANDLING AGENT
objection_handling_agent = Extractor(
    agent_name="objection_handling_agent",
    llm=nova_llm, 
    system_prompt=PromptHub().objection_handling_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: เบี้ยประกันเดือนละ 3,500 บาท แพงไปไหมครับ? เงินเดือนผมแค่ 25,000 บาท ต้องจ่ายค่าใช้จ่ายในบ้านอีกเยอะ ไม่รู้จะเหลือพอไหม
""".strip()

response = objection_handling_agent.run([dict(role="user", content=content)])
print("\nOBJECTION HANDLING AGENT:")
print(response.model_dump_json(indent=4))
objection_handling_agent.calculate_cost()


OBJECTION HANDLING AGENT:
{
    "action": "Acknowledge concern and reframe value",
    "explanation": "The customer is expressing concern about the price being too high relative to their budget. It's important to validate their concern and then reframe the value of the insurance to show how it provides long-term benefits.",
    "signals": [
        "Price objection raised",
        "Customer showing concern about affordability"
    ],
    "lines_to_say": [
        "ผมเข้าใจว่าคุณกังวลเรื่องค่าเบี้ยที่เกินรายได้ของคุณ แต่เรามีแผนที่คุ้มครองคุณในระยะยาวอย่างมีประโยชน์",
        "คุณมีคุณสมบัติที่ดีที่จะได้รับความคุ้มครองในสถานการณ์ที่สำคัญ และเราสามารถช่วยให้คุณคุมค่าได้อย่างมีประสิทธิภาพ"
    ]
}


{'input_tokens': 533,
 'output_tokens': 269,
 'input_cost': 0.000615615,
 'output_cost': 0.00124278,
 'total_cost': 0.001858395}